# Finnish Financial Sentiment Notebook - XLM-RoBERTa Training and Evaluation

## Imports

In [5]:
from datetime import datetime
import gc
import psutil

from google.colab import drive

from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    DataCollatorForLanguageModeling,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

from torch import nn
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score,
)

## Enable if upsampling is used 
# from sklearn.utils import resample 
from sklearn.model_selection import train_test_split
from datasets import Dataset
import pandas as pd
import datetime
import numpy as np
import torch

## Mount Google Drive

In [6]:
drive.mount('/content/drive')

Mounted at /content/drive


## GPU/RAM Check

In [7]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Sat Mar 21 07:29:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   31C    P0             47W /  600W |       3MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [8]:
ram_gb = psutil.virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 189.9 gigabytes of available RAM

You are using a high-RAM runtime!


In [9]:
timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
print(f"Current timestamp: {timestamp}")

Current timestamp: 2026-03-21_07-29-11


## Load Model

In [10]:
BASE_MODEL = 'FacebookAI/xlm-roberta-large'
FINNSENTIMENT_MODEL_PATH = f'/content/drive/MyDrive/ColabThesisData/models/{BASE_MODEL.split("/")[-1]}_finnsentiment_{timestamp}/'
NUM_LABELS  = 3
LABEL_NAMES = ["negative", "neutral", "positive"]
MAX_SEQ_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# ── Load base model for masked language modelling ─────────────────────────────
# Must use AutoModelForMaskedLM — the classification head is irrelevant for DAPT
base_model = AutoModelForMaskedLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    dtype=torch.bfloat16,
)

print(f"Base model loaded: {BASE_MODEL}")
print(f"Model device: {next(base_model.parameters()).device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

XLMRobertaForMaskedLM LOAD REPORT from: FacebookAI/xlm-roberta-large
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Base model loaded: FacebookAI/xlm-roberta-large
Model device: cuda:0


## Define Eval Func

In [11]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted", zero_division=0),
        "precision": precision_score(labels, preds, average="weighted", zero_division=0),
        "recall": recall_score(labels, preds, average="weighted", zero_division=0),
    }

## Domain-adapted pretraining

In [12]:
DAPT_SAMPLE_SEED = 42
DAPT_N_SAMPLES   = 100_000
DAPT_MODEL_PATH  = f'/content/drive/MyDrive/ColabThesisData/models/{BASE_MODEL.split("/")[-1]}_dapt_{timestamp}/'

# ── Sample unlabeled forum posts ──────────────────────────────────────────────
forum_posts = pd.read_parquet('/content/drive/MyDrive/ColabThesisData/cleaned_forum_posts.parquet')
dapt_df = (
    forum_posts[['message']]
    .dropna(subset=['message'])
    .sample(n=DAPT_N_SAMPLES, random_state=DAPT_SAMPLE_SEED)
    .reset_index(drop=True)
)
print(f"DAPT corpus: {len(dapt_df):,} posts (seed={DAPT_SAMPLE_SEED})")

# ── Tokenize ──────────────────────────────────────────────────────────────────
dapt_ds = Dataset.from_pandas(dapt_df)
dapt_ds = dapt_ds.map(
    lambda batch: tokenizer(batch['message'], truncation=True, max_length=MAX_SEQ_LENGTH),
    batched=True,
    remove_columns=['message'],
)

# ── Train ─────────────────────────────────────────────────────────────────────
dapt_training_args = TrainingArguments(
    output_dir='/tmp/dapt_checkpoints/',
    num_train_epochs=1,          # 1 epoch over 100k posts is standard for DAPT
    per_device_train_batch_size=16,
    learning_rate=3e-5,          # higher than fine-tuning — this is continued pre-training
    weight_decay=0.01,
    warmup_ratio=0.06,
    logging_steps=200,
    save_strategy="epoch",
    save_total_limit=1,
    bf16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=True,
)

dapt_trainer = Trainer(
    model=base_model,
    args=dapt_training_args,
    train_dataset=dapt_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=0.15,   # standard BERT masking rate
    ),
)

dapt_trainer.train()

dapt_trainer.save_model(DAPT_MODEL_PATH)
tokenizer.save_pretrained(DAPT_MODEL_PATH)
print(f"DAPT model saved to: {DAPT_MODEL_PATH}")

DAPT corpus: 100,000 posts (seed=42)


Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
200,1.836767
400,1.799090
600,1.782912
800,1.744987
1000,1.760924
1200,1.757976
1400,1.716858
1600,1.707275
1800,1.723282
2000,1.692685


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DAPT model saved to: /content/drive/MyDrive/ColabThesisData/models/xlm-roberta-large_dapt_2026-03-21_07-29-11/


## FinnSentiment Pre-training

In [13]:
finnsentiment = pd.read_pickle('/content/drive/MyDrive/ColabThesisData/finnsentiment_cleaned_unambiguous.pkl')

df_0 = finnsentiment[finnsentiment['label'] == 0]
df_1 = finnsentiment[finnsentiment['label'] == 1]
df_2 = finnsentiment[finnsentiment['label'] == 2]
df_1_balanced = df_1.sample(n=min(len(df_0), len(df_2)), random_state=42)
finnsentiment_balanced = pd.concat([df_0, df_1_balanced, df_2]).reset_index(drop=True)

print(f"FinnSentiment balanced: {len(finnsentiment_balanced):,}")
print(finnsentiment_balanced['label'].value_counts().sort_index())

FinnSentiment balanced: 3,818
label
0    1230
1    1230
2    1358
Name: count, dtype: int64


In [14]:
finnsentiment_balanced.sample(5)

,label,text
2562,2,Ylivoimaisesti osuvin kommentti.
1425,1,"Ja mulla ei koskaan ole ollut pentuja, en oo n..."
2186,1,5.4. 2017 klo 17.30 on Korpitien koululla huol...
3494,2,Kiitos paljon huojentavasta tiedosta!
3046,2,Hei pitkästä aikaa!


In [15]:
finnsentiment_balanced["label"].value_counts()

,count
label,
2,1358
0,1230
1,1230


In [16]:
dapt_model = AutoModelForSequenceClassification.from_pretrained(
    DAPT_MODEL_PATH,
    num_labels=NUM_LABELS,
    device_map="auto",
    dtype=torch.bfloat16,
)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/ColabThesisData/models/xlm-roberta-large_dapt_2026-03-21_07-29-11/
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
# 90/10 train/val split for FinnSentiment
fs_train_df, fs_val_df = train_test_split(
    finnsentiment_balanced, test_size=0.1, random_state=42,
    stratify=finnsentiment_balanced['label'],
)

def make_fs_dataset(df):
    hf = Dataset.from_pandas(df[['text', 'label']].reset_index(drop=True))
    return hf.map(
        lambda batch: tokenizer(batch['text'], truncation=True, max_length=MAX_SEQ_LENGTH),
        batched=True,
    )

fs_train_ds = make_fs_dataset(fs_train_df)
fs_val_ds   = make_fs_dataset(fs_val_df)

fs_training_args = TrainingArguments(
    output_dir='/tmp/fs_checkpoints/',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=75,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    bf16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=True,
)

fs_trainer = Trainer(
    model=dapt_model,
    args=fs_training_args,
    train_dataset=fs_train_ds,
    eval_dataset=fs_val_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

fs_trainer.train()

Map:   0%|          | 0/3436 [00:00<?, ? examples/s]

Map:   0%|          | 0/382 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.976891,0.450262,0.390673,0.656537,0.450262
2,No log,0.469236,0.890052,0.890413,0.891064,0.890052
3,0.889610,0.283572,0.908377,0.907754,0.909267,0.908377
4,0.889610,0.269221,0.905759,0.905472,0.905351,0.905759
5,0.360012,0.266689,0.905759,0.905472,0.905351,0.905759


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=1075, training_loss=0.6035625635191452, metrics={'train_runtime': 64.1848, 'train_samples_per_second': 267.665, 'train_steps_per_second': 16.749, 'total_flos': 2591695316751216.0, 'train_loss': 0.6035625635191452, 'epoch': 5.0})

In [18]:
fs_trainer.save_model(FINNSENTIMENT_MODEL_PATH)
tokenizer.save_pretrained(FINNSENTIMENT_MODEL_PATH)
print(f"FinnSentiment-tuned model saved to: {FINNSENTIMENT_MODEL_PATH}")

fs_results = fs_trainer.predict(fs_val_ds)
fs_preds = np.argmax(fs_results.predictions, axis=1)
ft_true = fs_val_df["label"].tolist()

print("\n" + "=" * 50)
print("HOLDOUT RESULTS — FINNSENTIMENT-TUNED MODEL")
print("=" * 50)
print(classification_report(ft_true, fs_preds, target_names=LABEL_NAMES))

print("Confusion matrix (rows=true, cols=predicted):")
print(pd.DataFrame(
    confusion_matrix(ft_true, fs_preds),
    index=[f"true_{l}" for l in LABEL_NAMES],
    columns=[f"pred_{l}" for l in LABEL_NAMES],
))

del dapt_model, fs_trainer
gc.collect(); torch.cuda.empty_cache()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

FinnSentiment-tuned model saved to: /content/drive/MyDrive/ColabThesisData/models/xlm-roberta-large_finnsentiment_2026-03-21_07-29-11/



HOLDOUT RESULTS — FINNSENTIMENT-TUNED MODEL
              precision    recall  f1-score   support

    negative       0.88      0.92      0.90       123
     neutral       0.92      0.84      0.88       123
    positive       0.92      0.96      0.94       136

    accuracy                           0.91       382
   macro avg       0.91      0.90      0.90       382
weighted avg       0.91      0.91      0.91       382

Confusion matrix (rows=true, cols=predicted):
               pred_negative  pred_neutral  pred_positive
true_negative            113             6              4
true_neutral              12           103              8
true_positive              3             3            130


## Pseudo-label Training

In [19]:
def make_hf_dataset(df):
    df = df.copy()
    df["text"] = "yritys: " + df["company_name"] + " viesti: " + df["message"]
    hf = Dataset.from_pandas(df[["text", "sentiment"]].rename(columns={"sentiment": "label"}).reset_index(drop=True))
    return hf.map(
        lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_SEQ_LENGTH),
        batched=True,
    )

In [20]:
PSEUDO_MODEL_PATH = f'/content/drive/MyDrive/ColabThesisData/models/{BASE_MODEL.split("/")[-1]}_pseudo_{timestamp}/'
LLM_LABELS_PATH   = '/content/drive/MyDrive/ColabThesisData/llm_labeled.parquet'

llm_labels = pd.read_parquet(LLM_LABELS_PATH)
print(f"LLM pseudo-labels: {len(llm_labels):,}")
print(llm_labels["sentiment"].value_counts().sort_index().rename({0: "negative", 1: "neutral", 2: "positive"}))

pseudo_ds = make_hf_dataset(llm_labels[["message", "sentiment", "company_name"]])

pseudo_model = AutoModelForSequenceClassification.from_pretrained(
    FINNSENTIMENT_MODEL_PATH, num_labels=NUM_LABELS, device_map="auto", dtype=torch.bfloat16)

pseudo_args = TrainingArguments(
    output_dir='/tmp/pseudo_ckpts/',
    num_train_epochs=10,
    per_device_train_batch_size=16,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.06,
    logging_steps=50,
    save_strategy="no",
    bf16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=True,
)

pseudo_trainer = Trainer(
    model=pseudo_model,
    args=pseudo_args,
    train_dataset=pseudo_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)
pseudo_trainer.train()

pseudo_trainer.save_model(PSEUDO_MODEL_PATH)
tokenizer.save_pretrained(PSEUDO_MODEL_PATH)
print(f"\nPseudo-label model saved to: {PSEUDO_MODEL_PATH}")

del pseudo_model, pseudo_trainer
gc.collect(); torch.cuda.empty_cache()

LLM pseudo-labels: 10,000
sentiment
negative    3591
neutral     2782
positive    3627
Name: count, dtype: int64


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
50,1.535040
100,1.151503
150,0.986210
200,0.954684
250,0.885906
300,0.907545
350,0.900038
400,0.923229
450,0.870978
500,0.891145


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Pseudo-label model saved to: /content/drive/MyDrive/ColabThesisData/models/xlm-roberta-large_pseudo_2026-03-21_07-29-11/


## Load Human-labeled Financial Data

In [21]:
human_labeled_sentiment = pd.read_parquet('/content/drive/MyDrive/ColabThesisData/labeled.parquet')

# Replace with your dataset loading
ds = human_labeled_sentiment

In [22]:
ds.sample(5)

,id,author_id,message,date_time,engagement,forum,url,company_name,ticker,message_length,year_month,year,sentiment,labeled_at
109,Sijoitustieto.comment-422,Sijoitustieto.Unknown,"Nordea on kyllä hyvä pankki, mutta minua huole...",2014-07-22 13:41:00.000,N/A,Sijoitustieto,https://www.sijoitustieto.fi/sijoituskeskustel...,Nordea,NDA-FI,255,2014-07,2014,0,2026-03-16T17:18:44.966540
10,Kauppalehti.post-7315597,Kauppalehti.58646,Jos markkinoilla ois sama näkemys kuin tällä p...,2023-12-15 15:53:46.000,N/A,Kauppalehti,https://keskustelu.kauppalehti.fi/threads/ssh-...,SSH Communications Security,SSH1V,133,2023-12,2023,0,2026-03-16T09:39:30.502368
184,Inderes.313187,Inderes.6530,Kyllähän Harvian käyttökatteesta näkee että hi...,2021-07-04 04:44:54.112,38,Inderes,https://forum.inderes.com/t/harvia-foorumi-eli...,Harvia,HARVIA,206,2021-07,2021,1,2026-03-16T17:41:56.446339
77,Kauppalehti.post-7616837,Kauppalehti.57519,Arvuuttelija70 sanoi:\nTämä on tällaista... os...,2025-08-26 12:12:27.000,"Reactions:\nVerot, öölman ja Arvuuttelija70",Kauppalehti,https://keskustelu.kauppalehti.fi/threads/endo...,Endomines Finland,PAMPALO,661,2025-08,2025,1,2026-03-16T14:58:37.056246
538,Sijoitustieto.comment-600,Sijoitustieto.Unknown,Näinhän se on :)....mullakin suurin huoli siin...,2014-08-07 13:51:00.000,N/A,Sijoitustieto,https://www.sijoitustieto.fi/sijoituskeskustel...,UPM-Kymmene,UPM,176,2014-08,2014,1,2026-03-16T20:47:03.335281


In [23]:
ds["sentiment"].value_counts()

,count
sentiment,
1,253
2,205
0,150


## K-Fold Cross Validation (Final Phase)

In [24]:
N_FOLDS = 5

cv_df = ds[["id", "message", "sentiment", "company_name"]].reset_index(drop=True)

kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

fold_results = []

for fold, (train_idx, val_idx) in enumerate(kf.split(cv_df, cv_df["sentiment"])):
    print(f"\n{'='*55}")
    print(f"  FOLD {fold+1}/{N_FOLDS}")
    print(f"{'='*55}")

    fold_train_df = cv_df.iloc[train_idx][["message", "sentiment", "company_name"]]
    fold_val_df   = cv_df.iloc[val_idx][["message", "sentiment", "company_name"]]
    fold_true     = fold_val_df["sentiment"].tolist()

    print(f"Train: {len(fold_train_df)}  |  Val: {len(fold_val_df)}")

    fold_train_ds = make_hf_dataset(fold_train_df)
    fold_val_ds   = make_hf_dataset(fold_val_df)

    fold_model = AutoModelForSequenceClassification.from_pretrained(
        PSEUDO_MODEL_PATH, num_labels=NUM_LABELS, device_map="auto", dtype=torch.bfloat16)

    fold_cw = compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=fold_train_df['sentiment'].values)
    fold_cw_tensor = torch.tensor(fold_cw, dtype=torch.float).to(fold_model.device)

    class FoldWeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            loss = nn.CrossEntropyLoss(weight=fold_cw_tensor)(outputs.logits, labels)
            return (loss, outputs) if return_outputs else loss

    fold_ft_args = TrainingArguments(
        output_dir=f'/tmp/fold_{fold+1}_ft_ckpts/',
        num_train_epochs=10,
        per_device_train_batch_size=16, per_device_eval_batch_size=32,
        learning_rate=3e-5, weight_decay=0.01, warmup_ratio=0.1, logging_steps=5,
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="f1", greater_is_better=True,
        bf16=torch.cuda.is_available(), report_to="none", remove_unused_columns=True,
    )

    fold_trainer = FoldWeightedTrainer(
        model=fold_model, args=fold_ft_args,
        train_dataset=fold_train_ds, eval_dataset=fold_val_ds, processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer), compute_metrics=compute_metrics,
    )
    fold_trainer.train()

    fold_preds = np.argmax(fold_trainer.predict(fold_val_ds).predictions, axis=1)
    fold_results.append({
        "accuracy":    accuracy_score(fold_true, fold_preds),
        "f1_weighted": f1_score(fold_true, fold_preds, average="weighted", zero_division=0),
        "f1_macro":    f1_score(fold_true, fold_preds, average="macro",    zero_division=0),
        "report":      classification_report(fold_true, fold_preds, target_names=LABEL_NAMES, output_dict=True, zero_division=0),
        "confusion":   confusion_matrix(fold_true, fold_preds),
    })
    print(f"  acc={fold_results[-1]['accuracy']:.3f}  f1_weighted={fold_results[-1]['f1_weighted']:.3f}  f1_macro={fold_results[-1]['f1_macro']:.3f}")

    del fold_model, fold_trainer
    gc.collect(); torch.cuda.empty_cache()


  FOLD 1/5
Train: 486  |  Val: 122


Map:   0%|          | 0/486 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

  acc=0.620  f1_weighted=0.619  f1_macro=0.619


In [25]:
accs = [r["accuracy"]    for r in fold_results]
f1w  = [r["f1_weighted"] for r in fold_results]
f1m  = [r["f1_macro"]    for r in fold_results]

print("── Per-fold Results ──")
for i, r in enumerate(fold_results):
    print(f"  Fold {i+1}: acc={r['accuracy']:.3f}  f1_weighted={r['f1_weighted']:.3f}  f1_macro={r['f1_macro']:.3f}")

print(f"\n── Mean ± Std ──")
print(f"  Accuracy    : {np.mean(accs):.3f} ± {np.std(accs):.3f}")
print(f"  F1 weighted : {np.mean(f1w):.3f} ± {np.std(f1w):.3f}")
print(f"  F1 macro    : {np.mean(f1m):.3f} ± {np.std(f1m):.3f}")

print(f"\n── Mean Per-class Metrics (across folds) ──")
for cls in LABEL_NAMES:
    prec = np.mean([r["report"][cls]["precision"] for r in fold_results])
    rec  = np.mean([r["report"][cls]["recall"]    for r in fold_results])
    f1   = np.mean([r["report"][cls]["f1-score"]  for r in fold_results])
    print(f"  {cls:>10}: precision={prec:.3f}  recall={rec:.3f}  f1={f1:.3f}")

agg_cm = sum(r["confusion"] for r in fold_results)
print(f"\n── Aggregated Confusion Matrix (all folds) ──")
print(pd.DataFrame(
    agg_cm,
    index=[f"true_{l}"  for l in LABEL_NAMES],
    columns=[f"pred_{l}" for l in LABEL_NAMES],
))

── Per-fold Results ──
  Fold 1: acc=0.615  f1_weighted=0.615  f1_macro=0.611
  Fold 2: acc=0.631  f1_weighted=0.630  f1_macro=0.612
  Fold 3: acc=0.615  f1_weighted=0.613  f1_macro=0.612
  Fold 4: acc=0.669  f1_weighted=0.666  f1_macro=0.669
  Fold 5: acc=0.620  f1_weighted=0.619  f1_macro=0.619

── Mean ± Std ──
  Accuracy    : 0.630 ± 0.021
  F1 weighted : 0.629 ± 0.019
  F1 macro    : 0.625 ± 0.022

── Mean Per-class Metrics (across folds) ──
    negative: precision=0.545  recall=0.647  f1=0.588
     neutral: precision=0.701  recall=0.585  f1=0.634
    positive: precision=0.632  recall=0.673  f1=0.651

── Aggregated Confusion Matrix (all folds) ──
               pred_negative  pred_neutral  pred_positive
true_negative             97            28             25
true_neutral              50           148             55
true_positive             30            37            138


## Final Model (All Data)

In [26]:
FINAL_MODEL_PATH = f'/content/drive/MyDrive/ColabThesisData/models/{BASE_MODEL.split("/")[-1]}_final_{timestamp}/'

all_human_df   = ds[["message", "sentiment", "company_name"]].copy()
final_train_ds = make_hf_dataset(all_human_df)

print(f"Final fine-tuning on {len(all_human_df):,} human-labeled samples")
print(all_human_df["sentiment"].value_counts().sort_index().rename({0: "negative", 1: "neutral", 2: "positive"}))

final_model = AutoModelForSequenceClassification.from_pretrained(
    PSEUDO_MODEL_PATH, num_labels=NUM_LABELS, device_map="auto", dtype=torch.bfloat16)

final_cw = compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=all_human_df['sentiment'].values)
final_cw_tensor = torch.tensor(final_cw, dtype=torch.float).to(final_model.device)

class FinalWeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = nn.CrossEntropyLoss(weight=final_cw_tensor)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

final_args = TrainingArguments(
    output_dir='/tmp/final_ckpts/',
    num_train_epochs=10,
    per_device_train_batch_size=16,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=5,
    save_strategy="no",
    bf16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=True,
)

_final_start = datetime.datetime.now()

final_trainer = FinalWeightedTrainer(
    model=final_model,
    args=final_args,
    train_dataset=final_train_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)
final_trainer.train()

_final_elapsed = datetime.datetime.now() - _final_start
print(f"Pipeline runtime: {str(_final_elapsed).split('.')[0]}")

final_trainer.save_model(FINAL_MODEL_PATH)
tokenizer.save_pretrained(FINAL_MODEL_PATH)
print(f"Final model saved to: {FINAL_MODEL_PATH}")

Map:   0%|          | 0/608 [00:00<?, ? examples/s]

Final fine-tuning on 608 human-labeled samples
sentiment
negative    150
neutral     253
positive    205
Name: count, dtype: int64


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
5,0.873466
10,1.118599
15,1.054583
20,1.045374
25,1.262450
30,0.835592
35,0.828078
40,0.944744
45,0.832283
50,0.876987


Pipeline runtime: 0:00:38


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final model saved to: /content/drive/MyDrive/ColabThesisData/models/xlm-roberta-large_final_2026-03-21_07-29-11/
